# Notebook 03: Failure Modes (Take-Home)

This notebook covers 5 ways agentic systems break — and how to fix them.

**Each section:** broken cell → explanation → fix → reflection question.

Work through these after lecture to deepen your understanding of agentic design.

In [1]:
import sys, os
from pathlib import Path
# Move to project root so relative data paths resolve correctly
_here = Path(os.getcwd())
if not (_here / 'data' / 'scenes').exists() and (_here.parent / 'data' / 'scenes').exists():
    os.chdir(_here.parent)
sys.path.insert(0, str(Path(os.getcwd())))

import json
from agent.scene import load_scene
from agent.types import BoundingBox
from agent import tools
from agent.schemas import TOOL_SCHEMAS
from agent.loop import run_agent, dispatch_tool, SYSTEM_PROMPT
from agent.mock import run_mock_agent
from unittest.mock import MagicMock

load_scene('scene_a')
print('Setup complete.')

Setup complete.


---

## Failure 1: Bad Tool Description

**Lesson:** Tool descriptions must be unambiguous. If a parameter name or description is unclear, the model may call the tool incorrectly or not at all.

In [2]:
# BROKEN: A schema with an ambiguous parameter name
broken_schema = {
    'name': 'compute_ndvi',
    'description': 'Compute vegetation index.',  # Vague — no guidance on when to use it
    'input_schema': {
        'type': 'object',
        'properties': {
            'area': {  # Renamed from 'region' — model will likely send 'area' not 'region'
                'type': 'object',
                'properties': {
                    'row_min': {'type': 'integer'},
                    'row_max': {'type': 'integer'},
                    'col_min': {'type': 'integer'},
                    'col_max': {'type': 'integer'},
                },
            }
        },
        'required': ['area'],
    },
}
print('Broken schema has parameter named: area')
print('Good schema has parameter named:', list(next(s for s in TOOL_SCHEMAS if s['name'] == 'compute_ndvi')['input_schema']['properties'].keys()))
print()
print('If the model is told the parameter is called "area" but dispatch_tool')
print('expects "region", the call will fail with a TypeError.')

Broken schema has parameter named: area
Good schema has parameter named: ['region']

If the model is told the parameter is called "area" but dispatch_tool
expects "region", the call will fail with a TypeError.


**Why it breaks:** `dispatch_tool` calls `tools.compute_ndvi(region=...)`. If the model sends `area=...` (as the schema dictates), Python raises `TypeError: unexpected keyword argument 'area'`.

**The fix:** Schema parameter names must exactly match the Python function parameter names. Descriptions should tell the model *when* to call the tool, not just *what* it does.

In [3]:
# FIXED: Show the correct schema for context
good_schema = next(s for s in TOOL_SCHEMAS if s['name'] == 'compute_ndvi')
print(json.dumps(good_schema, indent=2))
print()
print('Key fixes:')
print('  1. Parameter named "region" (matches Python function argument)')
print('  2. Detailed description tells model WHEN to use it and what output to expect')

{
  "name": "compute_ndvi",
  "description": "Compute NDVI (Normalized Difference Vegetation Index) for a region. NDVI = (NIR - Red) / (NIR + Red). Values near 1 = dense healthy vegetation; below 0.3 = stress, sparse vegetation, or bare soil. Returns mean, std, stressed-pixel fraction, and an image path.",
  "input_schema": {
    "type": "object",
    "properties": {
      "region": {
        "type": "object",
        "description": "Bounding box in pixel coordinates (integers).",
        "properties": {
          "row_min": {
            "type": "integer",
            "description": "Start row (inclusive)"
          },
          "row_max": {
            "type": "integer",
            "description": "End row (exclusive)"
          },
          "col_min": {
            "type": "integer",
            "description": "Start column (inclusive)"
          },
          "col_max": {
            "type": "integer",
            "description": "End column (exclusive)"
          }
        },
      

**Reflection question:** Look at the schema for `get_pixel_timeseries`. It uses `lat` and `lon` as parameter names even though these are actually pixel row/col coordinates. Is this a good naming choice? What are the tradeoffs? How might you fix it without breaking existing prompts?

---

## Failure 2: Exception vs. Error Result

**Lesson:** Tools that raise exceptions break the agent loop. Tools that return error results let the agent reason about the failure and adapt.

**Caveat:** Modern tools like Claude Code can handle exceptions from command line tools and programs,
and ultimately reason about or fix the issues anyway.
This is a more complex workflow however, and for bespoke hand-crafted agentic pipelines like the one
demonstrated here, it is often better to return error results with semantic info rather than the
Exception

In [4]:
# BROKEN: A version of compare_to_baseline that raises instead of returning error
def buggy_compare_to_baseline(region: BoundingBox, index: str):
    from agent import scene as _scene
    # This raises FileNotFoundError instead of returning a DiffResult
    base_nir = _scene.get_baseline_band('B08', region)  # raises for scene_b!
    return 'would compute diff here'

load_scene('scene_b')  # Scene B has no baseline
print('Calling buggy version on scene_b:')
try:
    buggy_compare_to_baseline(BoundingBox(0, 200, 0, 200), 'ndvi')
except FileNotFoundError as e:
    print(f'EXCEPTION RAISED: {e}')
    print()
    print('The agent loop receives an unhandled exception.')
    print('The loop crashes — no final report, no graceful degradation.')

Calling buggy version on scene_b:
EXCEPTION RAISED: No baseline available for the current scene. Scene 'scene_b' has has_baseline=False.

The agent loop receives an unhandled exception.
The loop crashes — no final report, no graceful degradation.


In [5]:
# FIXED: The actual implementation returns an error result
load_scene('scene_b')
result = tools.compare_to_baseline(BoundingBox(0, 200, 0, 200), index='ndvi')

print(f'success: {result.success}')
print(f'error_message: {result.error_message}')
print()
print('The agent receives this message in the tool_result block.')
print('It can then say: "No baseline available — I will rely on NDVI and NDWI instead."')

success: False
error_message: No baseline image found for this region. Cannot compute change. (No baseline available for the current scene. Scene 'scene_b' has has_baseline=False.)

The agent receives this message in the tool_result block.
It can then say: "No baseline available — I will rely on NDVI and NDWI instead."


**Reflection question:** Imagine a tool that calls an external weather API. Sometimes the API is unavailable (network error). Should this tool raise an exception or return an error result? What information should the error message contain to help the agent decide what to do next?

---

## Failure 3: Contradictory Indices

**Lesson:** Spectral indices can give conflicting signals. Without an explicit instruction to check for contradictions, the agent may confidently report based on only one index.

In [6]:
# Demonstrate contradictory indices using Scene B (mild uniform depression)
load_scene('scene_b')
ndvi_result = tools.compute_ndvi(BoundingBox(0, 200, 0, 200))
evi_result = tools.compute_evi(BoundingBox(0, 200, 0, 200))

print('NDVI result:')
print(ndvi_result.summary)
print()
print('EVI result:')
print(evi_result.summary)
print()
# Check if signals agree
ndvi_stressed = ndvi_result.low_fraction > 0.3
evi_stressed = evi_result.low_fraction > 0.3
if ndvi_stressed != evi_stressed:
    print('Contradiction: NDVI and EVI disagree on stress level!')
    print(f'   NDVI low-fraction: {ndvi_result.low_fraction:.1%}')
    print(f'   EVI  low-fraction: {evi_result.low_fraction:.1%}')
    print('   Scene B is designed to be ambiguous — this is expected.')
else:
    print('Indices agree on stress level.')

NDVI result:
NDVI for rows 0–200, cols 0–200: mean=0.556, std=0.057. Pixels below 0.3 (stressed): 0.0%. Vegetation appears generally healthy.

EVI result:
EVI for rows 0–200, cols 0–200: mean=0.424, std=0.071. Pixels below 0.2: 0.0%. EVI is less saturated than NDVI in dense canopy — use to verify NDVI findings.

Indices agree on stress level.


**Why this matters:** Without a system prompt instruction to 'check for contradictions', the agent might report based on whichever index it happened to call first.

**The fix:** Add an explicit instruction like:
```
If NDVI and EVI give contradictory signals (e.g., one suggests stress, the other does not),
explicitly note the contradiction in your report and express lower confidence.
```

Indeed, the best way to encode workflow information is in Agent Skills, which can detail and
preserve the specifics of workflows and make sure that any relevant internal knowledge is available
to the reasoning model.
For a bespoke pipeline, though, we can make sure that the explicit instruction is in the system
prompt, and it should have the same effect.

In [7]:
# Show the 'express uncertainty' instruction from the real system prompt
uncertainty_line = [line for line in SYSTEM_PROMPT.split('\n') if 'uncertainty' in line.lower() or 'conflicting' in line.lower()]
print('System prompt instructions about uncertainty:')
for line in uncertainty_line:
    print(f'  {line.strip()}')
print()
print('This is why the "express uncertainty" instruction is in the system prompt.')

System prompt instructions about uncertainty:
  7. Express uncertainty when appropriate. If indices give conflicting signals, say so.

This is why the "express uncertainty" instruction is in the system prompt.


**Reflection question:** The current system prompt says 'Express uncertainty when appropriate. If indices give conflicting signals, say so.' Is this specific enough? Write a stronger version of this instruction that tells the agent exactly what to do when NDVI and EVI disagree by more than 0.1.

---

## Failure 4: Infinite Loop Risk

**Lesson:** Without a `max_iterations` guard, a misbehaving tool or prompt could cause the agent to loop forever, burning API credits.

In [8]:
# Demonstrate: what happens if stop_reason is never 'end_turn'?
# We'll build a mock client that always returns tool_use

def make_always_tool_use_client():
    client = MagicMock()
    def always_tool_use(*args, **kwargs):
        mr = MagicMock()
        mr.stop_reason = 'tool_use'
        block = MagicMock()
        block.type = 'tool_use'
        block.id = 'mock_id'
        block.name = 'compute_ndvi'
        block.input = {'region': {'row_min': 0, 'row_max': 50, 'col_min': 0, 'col_max': 50}}
        mr.content = [block]
        return mr
    client.messages.create.side_effect = always_tool_use
    return client

load_scene('scene_a')
client = make_always_tool_use_client()
print('Running agent with max_iterations=3 and a stuck client...')
run_agent('Analyze.', client=client, max_iterations=3, verbose=True)
print(f'\nAPI calls made: {client.messages.create.call_count} (stopped at max_iterations=3)')

Running agent with max_iterations=3 and a stuck client...

[TOOL CALL] compute_ndvi
[INPUT] {
  "region": {
    "row_min": 0,
    "row_max": 50,
    "col_min": 0,
    "col_max": 50
  }
}
[RESULT] NDVI for rows 0–50, cols 0–50: mean=0.282, std=0.107. Pixels below 0.3 (stressed): 54.8%. ⚠️ Significant stressed area detected.

[TOOL CALL] compute_ndvi
[INPUT] {
  "region": {
    "row_min": 0,
    "row_max": 50,
    "col_min": 0,
    "col_max": 50
  }
}
[RESULT] NDVI for rows 0–50, cols 0–50: mean=0.282, std=0.107. Pixels below 0.3 (stressed): 54.8%. ⚠️ Significant stressed area detected.

[TOOL CALL] compute_ndvi
[INPUT] {
  "region": {
    "row_min": 0,
    "row_max": 50,
    "col_min": 0,
    "col_max": 50
  }
}
[RESULT] NDVI for rows 0–50, cols 0–50: mean=0.282, std=0.107. Pixels below 0.3 (stressed): 54.8%. ⚠️ Significant stressed area detected.


API calls made: 3 (stopped at max_iterations=3)


**The guard that saves us:** In `agent/loop.py`, the outer `for _ in range(max_iterations)` loop ensures we never exceed `max_iterations` API calls, even if the model never returns `end_turn`.

**What could cause infinite loops in practice?**
- A tool that always returns an error, causing the agent to retry forever
- A system prompt that says 'keep calling tools until you are certain' with no exit condition
- A model bug where `stop_reason` is never set to `end_turn`

In [9]:
# Show the guard in the actual loop code
import inspect
from agent.loop import run_agent
source = inspect.getsource(run_agent)
# Find the max_iterations line
for i, line in enumerate(source.split('\n')):
    if 'max_iterations' in line:
        print(f'Line {i+1}: {line}')
print()
print('The for loop ensures we stop. The warning at the end tells you when it triggers.')

Line 5:     max_iterations: int = 20,
Line 15:     for _ in range(max_iterations):
Line 56:         print(f"\nWarning: Hit max_iterations ({max_iterations}) without end_turn.")

The for loop ensures we stop. The warning at the end tells you when it triggers.


**Reflection question:** The current `max_iterations` default is 20. Is this the right value? What factors would you consider when choosing it for a production system? What would you log when the guard triggers to help with debugging?

---

## Failure 5: Overconfident Report

**Lesson:** Without explicit uncertainty instructions, agents tend to produce confident-sounding reports even when the data is ambiguous.

In [10]:
# Compare minimal vs full system prompt behavior with Scene B (ambiguous)
MINIMAL_PROMPT = """You are a crop health analyst. Assess the scene and produce a report."""

print('=== MINIMAL SYSTEM PROMPT ===')
print(MINIMAL_PROMPT)
print()
print('=== FULL SYSTEM PROMPT (key uncertainty instructions) ===')
for line in SYSTEM_PROMPT.split('\n'):
    if any(kw in line.lower() for kw in ['uncertainty', 'conflicting', 'uncertain', 'appropriate']):
        print(f'  {line.strip()}')

=== MINIMAL SYSTEM PROMPT ===
You are a crop health analyst. Assess the scene and produce a report.

=== FULL SYSTEM PROMPT (key uncertainty instructions) ===
  7. Express uncertainty when appropriate. If indices give conflicting signals, say so.


In [11]:
# Run mock agent to show the trace structure
# (Full comparison would need live API — we show the structural difference)
print('The minimal prompt omits:')
print('  - Instruction to explain reasoning before each tool call')
print('  - Instruction to only call tools when warranted')
print('  - Instruction to express uncertainty when appropriate')
print('  - Specific report structure requirements')
print()
print('Expected behavior difference with Scene B (ambiguous):')
print('  Minimal prompt -> confident report ("water stress detected") without caveats')
print('  Full prompt    -> hedged report ("possible mild stress, confidence LOW,"')
print('                   "recommend further investigation")')
print()
print('Try it yourself with ANTHROPIC_API_KEY set:')
print('  run_agent(request, system_prompt=MINIMAL_PROMPT)  # compare to default')

The minimal prompt omits:
  - Instruction to explain reasoning before each tool call
  - Instruction to only call tools when warranted
  - Instruction to express uncertainty when appropriate
  - Specific report structure requirements

Expected behavior difference with Scene B (ambiguous):
  Minimal prompt -> confident report ("water stress detected") without caveats
  Full prompt    -> hedged report ("possible mild stress, confidence LOW,"
                   "recommend further investigation")

Try it yourself with ANTHROPIC_API_KEY set:
  run_agent(request, system_prompt=MINIMAL_PROMPT)  # compare to default


**Reflection question:** Overconfident reports are particularly dangerous in high-stakes domains like agriculture, medicine, or infrastructure. Beyond system prompt instructions, what other mechanisms could you use to prevent overconfidence? Consider: output validation, human-in-the-loop triggers, confidence calibration, ensemble methods.

This is a thorny area and there are no industry-standard ways of handling the issue of how much or
how little to prompt an LLM.
The rule of thumb I use is that I need to concisely and precisely give the LLM everything it needs
to know about how I want it to operate, and nothing more.
This is easier said than done.
"Brevity is the soul of wit" could apply here.

---

## Summary: The 5 Failure Modes

| # | Failure | Fix |
|---|---------|-----|
| 1 | Bad tool description | Match parameter names to Python; write clear, specific descriptions |
| 2 | Exception vs. error result | Tools return error dicts; the loop never sees raw exceptions |
| 3 | Contradictory indices | Explicit system prompt instruction to flag and explain contradictions |
| 4 | Infinite loop | `max_iterations` guard; log when triggered |
| 5 | Overconfident report | Uncertainty instructions in system prompt; structural report requirements |

**The meta-lesson:** The agent loop itself is simple. Most failures come from the periphery — tool design, schema quality, and system prompt instructions. This is why each of these components deserves careful testing in isolation (Notebook 01) before combining them into an agent (Notebook 02).